In [2]:
%pip install pandas

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
import pandas as pd
import numpy as np

# =====================================================================
# HÀM FEATURE ENGINEERING (ĐÃ ĐỒNG BỘ TÊN FEATURES VÀ BỎ QUA CHU KỲ 30)
# =====================================================================
def advanced_feature_engineering(main_df, oil_df, holidays_df, stores_df):
    # 1. Tạo bản sao dữ liệu
    df = main_df.copy()
    
    # 2. Đồng bộ định dạng thời gian
    df['date'] = pd.to_datetime(df['date'])
    oil_df['date'] = pd.to_datetime(oil_df['date'])
    holidays_df['date'] = pd.to_datetime(holidays_df['date'])
    
    # 3. Merge với bảng Cửa hàng và Giá dầu
    df = df.merge(stores_df, on='store_nbr', how='left')
    df = df.merge(oil_df, on='date', how='left')
    
    # 4. Trích xuất các đặc trưng thời gian cơ bản
    df['year'] = df['date'].dt.year
    df['month'] = df['date'].dt.month
    df['day'] = df['date'].dt.day
    df['day_of_week'] = df['date'].dt.dayofweek  # 0: Thứ 2, 6: Chủ Nhật
    df['is_weekend'] = df['day_of_week'].isin([5, 6]).astype(int)
    
    # Đặc trưng ngày phát lương (wages) và cuối tháng
    df['is_payday'] = ((df['day'] == 15) | df['date'].dt.is_month_end).astype(int)
    df['is_month_end'] = df['date'].dt.is_month_end.astype(int)
    
    # Tạm thời gán mặc định cho Easter Week nếu dữ liệu chưa tính toán
    df['is_easter_week'] = 0 
    
    # 5. Xử lý ngày lễ (Holidays)
    real_holidays = holidays_df[holidays_df['transferred'] == False]
    national_hols = real_holidays[real_holidays['locale'] == 'National'].drop_duplicates(subset=['date'])
    
    df = df.merge(national_hols[['date', 'type']], on='date', how='left')
    
    # Đổi tên và xử lý cột ngày lễ quốc gia thành is_real_holiday
    if 'type_y' in df.columns:
        df.rename(columns={'type_y': 'holiday_national_type'}, inplace=True)
    elif 'type' in df.columns:
        df.rename(columns={'type': 'holiday_national_type'}, inplace=True)
        
    if 'holiday_national_type' in df.columns:
        df['is_real_holiday'] = df['holiday_national_type'].notna().astype(int)
        df['holiday_national_type'] = df['holiday_national_type'].fillna('None')
    else:
        df['is_real_holiday'] = 0
        df['holiday_national_type'] = 'None'
        
    # Đổi tên loại cửa hàng cho đúng chuẩn store_type của bạn
    if 'type_x' in df.columns:
        df.rename(columns={'type_x': 'store_type'}, inplace=True)
        
    # 6. Tạo các biến Trễ (Lag) và biến Trượt (Rolling) - CHỈ LẤY CHU KỲ 7 VÀ 14 NGÀY
    df.sort_values(['store_nbr', 'family', 'date'], inplace=True)
    
    # Dịch chuyển gốc 16 ngày để dự báo tập test tương lai (Tránh rò rỉ dữ liệu)
    base_lag = 16 
    
    if 'sales' in df.columns:
        # Tạo 3 biến Lag chính ứng với tên bạn yêu cầu
        df['sales_lag_1'] = df.groupby(['store_nbr', 'family'])['sales'].shift(base_lag)      # Gốc lag 16
        df['sales_lag_7'] = df.groupby(['store_nbr', 'family'])['sales'].shift(base_lag + 1)  # Gốc lag 17
        df['sales_lag_14'] = df.groupby(['store_nbr', 'family'])['sales'].shift(base_lag + 2) # Gốc lag 18
        
        # Chỉ tạo Rolling Mean và Std cho chu kỳ 7 và 14 ngày (Bỏ hoàn toàn 30 ngày)
        for window in [7, 14]:
            df[f'rolling_mean_{window}'] = (
                df.groupby(['store_nbr', 'family'])['sales_lag_1']
                .transform(lambda x: x.rolling(window).mean())
            )
            df[f'rolling_std_{window}'] = (
                df.groupby(['store_nbr', 'family'])['sales_lag_1']
                .transform(lambda x: x.rolling(window).std())
            )

    # 7. Mã hóa tự động các cột dạng chữ thành số (Label Encoding) ngoại trừ cột date
    dynamic_categorical_cols = df.select_dtypes(include=['object', 'category']).columns.tolist()
    if 'date' in dynamic_categorical_cols:
        dynamic_categorical_cols.remove('date')
        
    for col in dynamic_categorical_cols:
        df[col] = df[col].astype('category').cat.codes
            
    df = df.fillna(0)
    return df

In [4]:
import pandas as pd  # Thư viện dùng để xử lý dữ liệu dạng bảng (Dataframe)
import numpy as np   # Thư viện dùng để tính toán toán học và mảng số học

# ==========================================
# BUỚC 1: ĐỌC TẤT CẢ CÁC BẢNG DATA GỐC VÀ SẠCH
# ==========================================
print("🔄 Đang đọc dữ liệu...")  # In thông báo trạng thái đang tải file

# Đọc file chứa thông tin phân loại và khu vực của các cửa hàng
stores_df = pd.read_csv(r"D:\AIO26-Module1-Team_Level_Up-Store-Sales-Forecasting\data\data\stores.csv")

# Đọc file lịch trình các ngày lễ, sự kiện diễn ra tại Ecuador
holidays_df = pd.read_csv(r"D:\AIO26-Module1-Team_Level_Up-Store-Sales-Forecasting\data\data\holidays_events.csv")

# Đọc file dữ liệu biến động giá dầu hàng ngày (đã xử lý ffill)
oil_df = pd.read_csv(r"D:\AIO26-Module1-Team_Level_Up-Store-Sales-Forecasting\data\data\oil.csv") 

# Đọc tập dữ liệu huấn luyện lịch sử (chứa đầy đủ sales và transactions)
train_data = pd.read_csv(r"D:\AIO26-Module1-Team_Level_Up-Store-Sales-Forecasting\data\data\train.csv")

# Đọc tập dữ liệu cần dự đoán trong tương lai (không có cột sales)
test_data = pd.read_csv(r"D:\AIO26-Module1-Team_Level_Up-Store-Sales-Forecasting\data\data\test.csv")

# Chuyển đổi cột 'date' của tập train từ chuỗi văn bản (str) sang dạng thời gian thực tế (datetime)
train_data['date'] = pd.to_datetime(train_data['date'])

# Chuyển đổi cột 'date' của tập test từ chuỗi văn bản (str) sang dạng thời gian thực tế (datetime)
test_data['date'] = pd.to_datetime(test_data['date'])


# ==========================================
# BƯỚC 2: GỘP TRAIN VÀ TEST ĐỂ TÍNH LAG/ROLLING
# ==========================================
print("🔗 Đang gộp tập dữ liệu Train và Test...")  # In thông báo gộp bảng

# Xếp chồng tập train và test nối liền nhau để có bệ đỡ lịch sử liên tục khi dịch chuỗi thời gian
full_data = pd.concat([train_data, test_data], ignore_index=True)


# ==========================================
# BƯỚC 3: CHẠY HÀM ADVANCED FEATURE ENGINEERING
# ==========================================
print("🛠️ Đang tiến hành tạo các tính năng mới (Feature Engineering)...")  # In thông báo xử lý tính năng

# Gọi hàm advanced_feature_engineering đã viết để tự động tạo biến Lag, Rolling, Lễ, Tết... cho bảng full
full_featured = advanced_feature_engineering(full_data, oil_df, holidays_df, stores_df)


# ==========================================
# BƯỚC 4: TÁCH LẠI THÀNH TRAIN VÀ TEST SẠCH + LỌC FEATURE
# ==========================================
print("✂️ Đang tách dữ liệu và dọn dẹp biến thừa...")  # In thông báo phân rã bảng

# Định nghĩa danh sách 22 cột đặc trưng chuẩn mong muốn được giữ lại
features_to_keep = [
    'date', 'family', 'sales', 'transactions', 'is_real_holiday', 'store_type', 'cluster', 
    'is_weekend', 'is_payday', 'is_month_end', 'is_easter_week', 'day_of_week', 'year', 
    'month', 'day', 'sales_lag_1', 'sales_lag_7', 'sales_lag_14', 
    'rolling_mean_7', 'rolling_std_7', 'rolling_mean_14', 'rolling_std_14'
]

# Tìm ngày bắt đầu nhỏ nhất của tập test (thường là ngày 16/08/2017) để làm mốc phân chia
test_data_start = test_data['date'].min()


# --- XỬ LÝ VÀ CHỌN LỌC TẬP TEST ---
# Lọc lấy các dòng thuộc giai đoạn tương lai (từ ngày mốc test trở đi) và copy ra bảng độc lập
test_featured = full_featured[full_featured['date'] >= test_data_start].copy()

# Tạo danh sách các cột cần giữ cho tập test (lọc từ list chuẩn nhưng loại bỏ 'sales' và 'transactions' vì tương lai chưa có)
test_cols = [col for col in features_to_keep if col in test_featured.columns and col not in ['sales', 'transactions']]

# Tiến hành ép bảng test chỉ giữ lại đúng các cột trong danh sách vừa chọn lọc ở trên
test_featured = test_featured[test_cols]


# --- XỬ LÝ VÀ CHỌN LỌC TẬP TRAIN ---
# Lọc lấy các dòng thuộc giai đoạn lịch sử (trước ngày mốc tập test) và copy ra bảng độc lập
train_featured = full_featured[full_featured['date'] < test_data_start].copy()

# Xóa bỏ các dòng của những ngày đầu năm 2013 bị trống giá trị (NaN) do cơ chế dịch chuỗi lag 16 ngày gây ra
train_featured.dropna(subset=['sales_lag_1'], inplace=True) 

# Tạo danh sách các cột cần giữ cho tập train (lọc những cột có tồn tại thực tế trong bảng full_featured)
train_cols = [col for col in features_to_keep if col in train_featured.columns]

# Tiến hành ép bảng train chỉ giữ lại đúng các cột đặc trưng mong muốn (bao gồm cả sales và transactions)
train_featured = train_featured[train_cols]


# ==========================================
# BƯỚC 5: LƯU LẠI THÀNH FILE SẴN SÀNG CHO MODEL
# ==========================================
print("💾 Đang lưu dữ liệu sạch ra file...")  # In thông báo ghi file ra ổ đĩa

# Lưu dữ liệu Train sạch ra file CSV, bỏ cột chỉ mục index thừa của pandas để giảm dung lượng file
train_featured.to_csv('train_ready_to_model.csv', index=False)

# Lưu dữ liệu Test sạch ra file CSV, bỏ cột chỉ mục index thừa của pandas phục vụ cho dự báo sau này
test_featured.to_csv('test_ready_to_model.csv', index=False)

# In số lượng dòng và cột của tập Train sau xử lý để kiểm tra tính toàn vẹn dữ liệu
print(f"👉 Shape tập Train mới (Có đầy đủ Sales & Transactions): {train_featured.shape}")

# In số lượng dòng và cột của tập Test sau xử lý xem đã tinh gọn chuẩn xác chưa
print(f"👉 Shape tập Test mới (Tinh gọn phục vụ Predict): {test_featured.shape}")

🔄 Đang đọc dữ liệu...
🔗 Đang gộp tập dữ liệu Train và Test...
🛠️ Đang tiến hành tạo các tính năng mới (Feature Engineering)...


C:\Users\TIN\AppData\Local\Temp\ipykernel_14084\2848012871.py:81: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  dynamic_categorical_cols = df.select_dtypes(include=['object', 'category']).columns.tolist()


✂️ Đang tách dữ liệu và dọn dẹp biến thừa...
💾 Đang lưu dữ liệu sạch ra file...
👉 Shape tập Train mới (Có đầy đủ Sales & Transactions): (3000888, 21)
👉 Shape tập Test mới (Tinh gọn phục vụ Predict): (28512, 20)


In [8]:
train = pd.read_csv(r"D:\AIO26-Module1-Team_Level_Up-Store-Sales-Forecasting\data\train_ready_to_model.csv")

test = pd.read_csv(r"D:\AIO26-Module1-Team_Level_Up-Store-Sales-Forecasting\data\test_ready_to_model.csv")


print(train.head())


         date  family  sales  is_real_holiday  store_type  cluster  \
0  2013-01-01       0    0.0                1           3       13   
1  2013-01-02       0    2.0                0           3       13   
2  2013-01-03       0    3.0                0           3       13   
3  2013-01-04       0    3.0                0           3       13   
4  2013-01-05       0    5.0                1           3       13   

   is_weekend  is_payday  is_month_end  is_easter_week  ...  year  month  day  \
0           0          0             0               0  ...  2013      1    1   
1           0          0             0               0  ...  2013      1    2   
2           0          0             0               0  ...  2013      1    3   
3           0          0             0               0  ...  2013      1    4   
4           1          0             0               0  ...  2013      1    5   

   sales_lag_1  sales_lag_7  sales_lag_14  rolling_mean_7  rolling_std_7  \
0          0.0  

<class 'pandas.DataFrame'>
RangeIndex: 28512 entries, 0 to 28511
Data columns (total 20 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   date             28512 non-null  str    
 1   family           28512 non-null  int64  
 2   is_real_holiday  28512 non-null  int64  
 3   store_type       28512 non-null  int64  
 4   cluster          28512 non-null  int64  
 5   is_weekend       28512 non-null  int64  
 6   is_payday        28512 non-null  int64  
 7   is_month_end     28512 non-null  int64  
 8   is_easter_week   28512 non-null  int64  
 9   day_of_week      28512 non-null  int64  
 10  year             28512 non-null  int64  
 11  month            28512 non-null  int64  
 12  day              28512 non-null  int64  
 13  sales_lag_1      28512 non-null  float64
 14  sales_lag_7      28512 non-null  float64
 15  sales_lag_14     28512 non-null  float64
 16  rolling_mean_7   28512 non-null  float64
 17  rolling_std_7    28512 